# Kaggle job: setup-regions

Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment

Inputs are resolved from the declared resource graph. No raw dataset re-upload is required.

In [1]:
import os,json
from pathlib import Path
_SETUP_CONFIG = {'profile': 'regions', 'action': 'prepare', 'build_wheelhouse': True, 'wheelhouse': '/kaggle/working/setup-regions/wheelhouse', 'target': '/tmp/vpi-runtime-setup-regions', 'output_root': '/kaggle/working/setup-regions', 'download_models': False}
_SETUP_PATH = Path('/kaggle/working/setup-config.json')
_SETUP_PATH.write_text(json.dumps(_SETUP_CONFIG))
os.environ['VPI_NEXT_CONFIG'] = str(_SETUP_PATH)

# Current Kaggle runtime setup

This setup-only notebook inventories the actual Linux kernel and prepares one isolated CPU, DINO, LocateAnything/SAM or publication environment. It never opens a scene, anomaly label or scientific evaluator and never calls a model. It preserves exact package pins while recording the observed Python 3.12 patch before the new method freeze.

The source remains a verified working-directory snapshot. Use a separate job for each profile. CPU setup does not need a GPU; CUDA wheels can also be prepared without allocating one. Region inference later requires the configured T4 accelerator. Retain the wheelhouse and runtime-profile output before starting downstream stages.

In [2]:
from pathlib import Path
import json, os
try:
    NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb'
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)

    SOURCE = _runtime.verify_source(CODE_ROOT, NOTEBOOK_RELATIVE)
    sys.path.insert(0, str(CODE_ROOT / 'scripts'))
    print({'source_snapshot_sha256': SOURCE['snapshot_sha256'], 'scientific_imports_started': False})
except BaseException as error:
    receipt_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    receipt_root.mkdir(parents=True, exist_ok=True)
    temporary = receipt_root/'setup-bootstrap.json.partial'
    temporary.write_text(json.dumps({'status':'setup_bootstrap_failed','error_type':type(error).__name__,'scientific_run_started':False})+'\n')
    os.replace(temporary, receipt_root/'setup-bootstrap.json')
    raise

{'source_snapshot_sha256': 'ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf', 'scientific_imports_started': False}


## Reviewed setup configuration

Set `VPI_NEXT_CONFIG` to a setup JSON containing `profile` (`cpu`, `dino`, `regions` or `tracking`), `action` (`probe` or `prepare`), `output_root`, `target`, and `wheelhouse`. Set `build_wheelhouse=true` to build the exact offline artifacts on Linux from the source lock. The default action only records metadata. Retained outputs belong under `/kaggle/working`; dependency-only targets use fresh `/tmp/vpi-runtime-*` directories so thousands of installed files are not retained as notebook output. GPU and DINO use separate targets because their Torch/Transformers versions differ.

In [3]:
CONFIG = _runtime.resolve_config('runtime-setup', CODE_ROOT)
PROFILE_NAMES = {'cpu': 'linux-cpu-cache-cp312', 'dino': 'linux-dino-fp32-cpu', 'regions': 'linux-regions-cuda126', 'tracking': 'linux-tracking-cp312'}
if CONFIG.get('profile') not in PROFILE_NAMES or CONFIG.get('action', 'probe') not in {'probe', 'prepare'}:
    raise ValueError('Choose one registered profile and probe/prepare action')
profile_path = CODE_ROOT / 'requirements/profiles' / (PROFILE_NAMES[CONFIG['profile']] + '.json')
output_root = _runtime.writable_output(CONFIG['output_root'])
from kaggle_setup_probe import probe
PROBE = probe([profile_path])
_runtime.atomic_json(output_root / 'initial_setup_probe.json', PROBE)
print(PROBE)

{'schema': 'vpi-kaggle-setup-probe/v1', 'status': 'metadata_probe_complete', 'system': 'Linux', 'machine': 'x86_64', 'python': '3.12.13', 'packages': {'Pillow': '11.3.0', 'accelerate': '1.13.0', 'contourpy': '1.3.3', 'cycler': '0.12.1', 'decord': None, 'einops': '0.8.2', 'fonttools': '4.62.1', 'huggingface-hub': '1.11.0', 'hydra-core': None, 'iopath': None, 'kiwisolver': '1.5.0', 'lmdb': None, 'matplotlib': '3.10.0', 'numpy': '2.0.2', 'opencv-python-headless': '4.13.0.92', 'packaging': '26.1', 'pandas': '2.3.3', 'peft': '0.19.1', 'pyparsing': '3.3.2', 'python-dateutil': '2.9.0.post0', 'pytz': '2025.2', 'scipy': '1.16.3', 'six': '1.17.0', 'timm': '1.0.26', 'torch': '2.10.0+cpu', 'torchvision': '0.25.0+cpu', 'tqdm': '4.67.3', 'transformers': '5.0.0', 'tzdata': '2026.1'}, 'image': {'git_commit': 'fc61d5cda7da39530055bae9bd0e92865f995cd9', 'build_date': '20260629-122507'}, 'gpu': {'available': False}, 'profiles': {'linux-regions-cuda126.json': {'python_matches': False, 'package_mismatches'

## Build verified wheels and private runtime

Wheel builds are bounded and use official PyPI/PyTorch indexes. Only dependencies are resolved or built; nothing is installed into the notebook kernel. The next step installs the verified wheelhouse into a fresh private target and seals its exact package metadata and observed Python patch. This is setup verification, not scientific or model compatibility evidence.

In [4]:
if CONFIG.get('action', 'probe') == 'prepare':
    wheelhouse = Path(CONFIG['wheelhouse'])
    if CONFIG.get('build_wheelhouse', False):
        wheelhouse = _runtime.writable_output(wheelhouse)
        if CONFIG['profile'] == 'cpu':
            from prepare_linux_wheelhouse import prepare as build_wheels
        else:
            from prepare_neural_wheelhouse import prepare as build_wheels
        print(build_wheels(profile_path, wheelhouse))
    from prepare_runtime_environment import prepare_isolated
    base_profile = CODE_ROOT / 'requirements/current-evaluation-runtime.json' if CONFIG['profile'] == 'cpu' else profile_path
    RECEIPT = prepare_isolated(base_profile, wheelhouse, _runtime.runtime_target(CONFIG['target']), output_root / 'runtime-profile')
    print(RECEIPT)
else:
    print({'status': 'metadata_only', 'dependencies_installed': False})

{'status': 'resolved_wheelhouse_prepared', 'elapsed_seconds': 215.09808593599996, 'observed_python': '3.12.13', 'profile_python': '3.12.14', 'python_patch_matches': False, 'wheel_count': 70, 'total_bytes': 3775918937, 'models_loaded': False, 'datasets_opened': False, 'kernel_packages_changed': False, 'target_imports_verified': False}
{'status': 'runtime_profile_prepared', 'python': '3.12.13', 'profile_sha256': '063cb76409c3480754e01431c76421094df2a6864094134acee25393e05a2e18', 'private_target_prepared': True, 'models_loaded': False, 'datasets_opened': False, 'scientific_methods_executed': False, 'isolated_child_process': True}


## Optional pinned model assets

Set `download_models=true` only in a DINO or regions setup job, plus `model_asset_root` and `models_only=all|regions|dino`. The bounded model downloader uses the existing immutable model revisions and checkpoint hashes. The outer timeout covers each selected model's 1,800-second download budget plus checksum time; timeout cleanup stops the whole child process group. It downloads assets without constructing or calling a model. Reuse its output in inference jobs.

In [5]:
if CONFIG.get('download_models', False):
    if CONFIG.get('action') != 'prepare' or CONFIG['profile'] not in {'dino', 'regions'}:
        raise ValueError('Model asset download requires a prepared DINO or regions setup profile')
    model_root = _runtime.writable_output(CONFIG['model_asset_root'])
    print(_runtime.run_model_asset_preparation(CODE_ROOT, model_root, CONFIG.get('models_only', CONFIG['profile']), CONFIG['target'], per_model_timeout=1800))
else:
    print({'model_asset_download': 'not requested'})

{'model_asset_download': 'not requested'}


## Downstream binding

A downstream job sets `VPI_RUNTIME_PROFILE_ROOT` to the sealed `runtime-profile` resource, `VPI_WHEELHOUSE` to the matching wheelhouse, `VPI_PACKAGE_TARGET` to a fresh target and `VPI_INSTALL_DEPENDENCIES=1` before its normal bootstrap. DINO also sets `VPI_BASE_RUNTIME_PROFILE=requirements/profiles/linux-dino-fp32-cpu.json`; the region notebook selects its GPU profile explicitly.

The bootstrap verifies unchanged package/device pins, the base-profile hash and the observed Linux Python patch. The subsequent method freeze records the actual scientific runtime. A later runtime change cannot bypass a completed freeze. Run development compatibility checks before protected evaluation. Pinned model assets may be prepared by the optional bounded download cell above; no model inference is run.

In [6]:
from current_launch import complete_job
JOB_COMPLETION = complete_job({'schema': 'vpi-current-kaggle-launch-plan/v1', 'models_fixed': True, 'raw_dataset_reupload_required': False, 'original_roles': {'development': 1000, 'calibration': 600, 'internal_lock': 400}, 'official_roles': {'seen': 715, 'unseen': 2437}, 'methods': ['winner_cluster_consensus', 'object_bank_top2'], 'core_views': [2, 3, 5], 'diagnostic_views': [1], 'schedule_count': 19, 'resources': {'source': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-current-evaluation-source', 'marker': 'prepared_source_manifest.json', 'suffixes': ['source', ''], 'availability': 'prepare_upload'}, 'support': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-current-evaluation-support', 'marker': 'support_manifest.json', 'suffixes': [''], 'availability': 'prepare_upload'}, 'raw': {'kind': 'dataset', 'ref': 'esreekarreddy/toysad8k-full-private-eda', 'marker': 'package-checksums.sha256', 'suffixes': ['', 'toysad8k-full-private-eda'], 'availability': 'existing_kaggle', 'contains_outcomes': True}, 'selection': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-toysad-selection-private', 'marker': 'experiment_roles.jsonl', 'suffixes': ['toysad8k-train-2000-v2'], 'availability': 'existing_kaggle'}, 'administrative': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-toysad-selection-private', 'marker': 'restricted_source_map.jsonl', 'suffixes': ['toysad8k-train-2000-v2-administrative-vault'], 'availability': 'existing_kaggle'}, 'rgb-screen': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-toysad-ds1-screen100-model-input', 'marker': 'package_record.json', 'suffixes': ['model-input', ''], 'availability': 'existing_kaggle'}, 'rgb-remainder': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-toysad-development-remainder-model-input', 'marker': 'package_record.json', 'suffixes': ['development-remainder/model-input', 'model-input', ''], 'availability': 'existing_kaggle'}, 'rgb-confirm': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-toysad-ds2-confirmation-rgb', 'marker': 'package_record.json', 'suffixes': ['model-input', ''], 'availability': 'existing_kaggle'}, 'development-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/toysad-8k-frozen-development-pipeline', 'marker': 'full-development-lock.json', 'suffixes': ['toysad8k_frozen_development_pipeline'], 'required_sha256': {'full-development-lock.json': '9d8505d2bccdcf9d6e59ebf39c52676278517ff6564c391aa88cb4fb113c050c'}, 'availability': 'existing_kaggle'}, 'reuse100': {'kind': 'dataset', 'ref': 'esreekarreddy/vpi-current-development100-feature-reuse', 'marker': 'feature_shard_lock.json', 'suffixes': [''], 'availability': 'prepared_local_upload'}, 'runtime-cpu': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-cpu', 'producer_job': 'setup-cpu', 'marker': 'runtime_profile_lock.json', 'suffixes': ['setup-cpu/runtime-profile']}, 'wheels-cpu': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-cpu', 'producer_job': 'setup-cpu', 'marker': 'wheelhouse_manifest.json', 'suffixes': ['setup-cpu/wheelhouse']}, 'runtime-dino': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-dino', 'producer_job': 'setup-dino', 'marker': 'runtime_profile_lock.json', 'suffixes': ['setup-dino/runtime-profile']}, 'wheels-dino': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-dino', 'producer_job': 'setup-dino', 'marker': 'wheelhouse_manifest.json', 'suffixes': ['setup-dino/wheelhouse']}, 'runtime-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-regions', 'producer_job': 'setup-regions', 'marker': 'runtime_profile_lock.json', 'suffixes': ['setup-regions/runtime-profile']}, 'wheels-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-regions', 'producer_job': 'setup-regions', 'marker': 'wheelhouse_manifest.json', 'suffixes': ['setup-regions/wheelhouse']}, 'runtime-tracking': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-tracking', 'producer_job': 'setup-tracking', 'marker': 'runtime_profile_lock.json', 'suffixes': ['setup-tracking/runtime-profile']}, 'wheels-tracking': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-tracking', 'producer_job': 'setup-tracking', 'marker': 'wheelhouse_manifest.json', 'suffixes': ['setup-tracking/wheelhouse']}, 'models': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-setup-dino', 'producer_job': 'setup-dino', 'marker': 'model_assets_receipt.json', 'suffixes': ['setup-dino/model-assets']}, 'freeze': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-freeze', 'producer_job': 'freeze', 'marker': 'method_freeze.json', 'suffixes': ['freeze']}, 'development-index': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-index', 'producer_job': 'development-index', 'marker': 'rgb_index.json', 'suffixes': ['development-index']}, 'development-geometry': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-geometry', 'producer_job': 'development-geometry', 'marker': 'geometry_manifest.json', 'suffixes': ['development-geometry']}, 'calibration-index': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-index', 'producer_job': 'calibration-index', 'marker': 'rgb_index.json', 'suffixes': ['calibration-index']}, 'calibration-geometry': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-geometry', 'producer_job': 'calibration-geometry', 'marker': 'geometry_manifest.json', 'suffixes': ['calibration-geometry']}, 'internal_lock-index': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-index', 'producer_job': 'internal_lock-index', 'marker': 'rgb_index.json', 'suffixes': ['internal_lock-index']}, 'internal_lock-geometry': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-geometry', 'producer_job': 'internal_lock-geometry', 'marker': 'geometry_manifest.json', 'suffixes': ['internal_lock-geometry']}, 'seen-index': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-index', 'producer_job': 'seen-index', 'marker': 'rgb_index.json', 'suffixes': ['seen-index']}, 'seen-geometry': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-geometry', 'producer_job': 'seen-geometry', 'marker': 'geometry_manifest.json', 'suffixes': ['seen-geometry']}, 'unseen-index': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-index', 'producer_job': 'unseen-index', 'marker': 'rgb_index.json', 'suffixes': ['unseen-index']}, 'unseen-geometry': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-geometry', 'producer_job': 'unseen-geometry', 'marker': 'geometry_manifest.json', 'suffixes': ['unseen-geometry']}, 'official-exposure': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-official-exposure', 'producer_job': 'official-exposure', 'marker': 'official_exposure_ledger.json', 'suffixes': ['official-exposure']}, 'development-neutral': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-neutral', 'producer_job': 'development-neutral', 'marker': 'neutral_correspondence.json', 'suffixes': ['development-neutral']}, 'development-cache-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-00', 'producer_job': 'development-cache-00', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-00']}, 'development-cache-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-01', 'producer_job': 'development-cache-01', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-01']}, 'development-cache-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-02', 'producer_job': 'development-cache-02', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-02']}, 'development-cache-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-03', 'producer_job': 'development-cache-03', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-03']}, 'development-cache-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-04', 'producer_job': 'development-cache-04', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-04']}, 'development-cache-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-05', 'producer_job': 'development-cache-05', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-05']}, 'development-cache-06': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-06', 'producer_job': 'development-cache-06', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-06']}, 'development-cache-07': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-07', 'producer_job': 'development-cache-07', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-07']}, 'development-cache-08': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-08', 'producer_job': 'development-cache-08', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-08']}, 'development-cache-09': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache-09', 'producer_job': 'development-cache-09', 'marker': 'cache_shard_lock.json', 'suffixes': ['development-cache-09']}, 'development-cache': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-cache', 'producer_job': 'development-cache', 'marker': 'cohort_manifest.json', 'suffixes': ['development-cache']}, 'development-predict': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-predict', 'producer_job': 'development-predict', 'marker': 'prediction_lock.json', 'suffixes': ['development-predict']}, 'development-truth': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-truth', 'producer_job': 'development-truth', 'marker': 'current_evaluator_manifest.json', 'suffixes': ['development-truth'], 'contains_outcomes': True}, 'development-evaluate': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-development-evaluate', 'producer_job': 'development-evaluate', 'marker': 'evaluation_lock.json', 'suffixes': ['development-evaluate'], 'contains_outcomes': True}, 'calibration-rgb': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-rgb', 'producer_job': 'calibration-rgb', 'marker': 'package_record.json', 'suffixes': ['calibration-rgb/model-input']}, 'calibration-regions-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions-00', 'producer_job': 'calibration-regions-00', 'marker': 'region_shard_lock.json', 'suffixes': ['calibration-regions-00']}, 'calibration-regions-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions-01', 'producer_job': 'calibration-regions-01', 'marker': 'region_shard_lock.json', 'suffixes': ['calibration-regions-01']}, 'calibration-regions-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions-02', 'producer_job': 'calibration-regions-02', 'marker': 'region_shard_lock.json', 'suffixes': ['calibration-regions-02']}, 'calibration-regions-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions-03', 'producer_job': 'calibration-regions-03', 'marker': 'region_shard_lock.json', 'suffixes': ['calibration-regions-03']}, 'calibration-regions-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions-04', 'producer_job': 'calibration-regions-04', 'marker': 'region_shard_lock.json', 'suffixes': ['calibration-regions-04']}, 'calibration-regions-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions-05', 'producer_job': 'calibration-regions-05', 'marker': 'region_shard_lock.json', 'suffixes': ['calibration-regions-05']}, 'calibration-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-regions', 'producer_job': 'calibration-regions', 'marker': 'artifact_lock.json', 'suffixes': ['calibration-regions']}, 'calibration-neutral': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-neutral', 'producer_job': 'calibration-neutral', 'marker': 'neutral_correspondence.json', 'suffixes': ['calibration-neutral']}, 'calibration-cache-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache-00', 'producer_job': 'calibration-cache-00', 'marker': 'cache_shard_lock.json', 'suffixes': ['calibration-cache-00']}, 'calibration-cache-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache-01', 'producer_job': 'calibration-cache-01', 'marker': 'cache_shard_lock.json', 'suffixes': ['calibration-cache-01']}, 'calibration-cache-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache-02', 'producer_job': 'calibration-cache-02', 'marker': 'cache_shard_lock.json', 'suffixes': ['calibration-cache-02']}, 'calibration-cache-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache-03', 'producer_job': 'calibration-cache-03', 'marker': 'cache_shard_lock.json', 'suffixes': ['calibration-cache-03']}, 'calibration-cache-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache-04', 'producer_job': 'calibration-cache-04', 'marker': 'cache_shard_lock.json', 'suffixes': ['calibration-cache-04']}, 'calibration-cache-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache-05', 'producer_job': 'calibration-cache-05', 'marker': 'cache_shard_lock.json', 'suffixes': ['calibration-cache-05']}, 'calibration-cache': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-cache', 'producer_job': 'calibration-cache', 'marker': 'cohort_manifest.json', 'suffixes': ['calibration-cache']}, '08f1': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-08f1', 'producer_job': '08f1', 'marker': 'cohort_manifest.json', 'suffixes': ['08f1/model-cache']}, '09a': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-09a', 'producer_job': '09a', 'marker': 'prediction_lock.json', 'suffixes': ['09a']}, 'calibration-truth': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-calibration-truth', 'producer_job': 'calibration-truth', 'marker': 'current_evaluator_manifest.json', 'suffixes': ['calibration-truth'], 'contains_outcomes': True}, '09b': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-09b', 'producer_job': '09b', 'marker': 'evaluation_lock.json', 'suffixes': ['09b'], 'contains_outcomes': True}, 'internal_lock-rgb': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-rgb', 'producer_job': 'internal_lock-rgb', 'marker': 'package_record.json', 'suffixes': ['internal_lock-rgb/model-input']}, 'internal_lock-regions-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-00', 'producer_job': 'internal_lock-regions-00', 'marker': 'region_shard_lock.json', 'suffixes': ['internal_lock-regions-00']}, 'internal_lock-regions-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-01', 'producer_job': 'internal_lock-regions-01', 'marker': 'region_shard_lock.json', 'suffixes': ['internal_lock-regions-01']}, 'internal_lock-regions-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-02', 'producer_job': 'internal_lock-regions-02', 'marker': 'region_shard_lock.json', 'suffixes': ['internal_lock-regions-02']}, 'internal_lock-regions-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-03', 'producer_job': 'internal_lock-regions-03', 'marker': 'region_shard_lock.json', 'suffixes': ['internal_lock-regions-03']}, 'internal_lock-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions', 'producer_job': 'internal_lock-regions', 'marker': 'artifact_lock.json', 'suffixes': ['internal_lock-regions']}, 'internal_lock-neutral': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-neutral', 'producer_job': 'internal_lock-neutral', 'marker': 'neutral_correspondence.json', 'suffixes': ['internal_lock-neutral']}, 'internal_lock-cache-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-00', 'producer_job': 'internal_lock-cache-00', 'marker': 'cache_shard_lock.json', 'suffixes': ['internal_lock-cache-00']}, 'internal_lock-cache-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-01', 'producer_job': 'internal_lock-cache-01', 'marker': 'cache_shard_lock.json', 'suffixes': ['internal_lock-cache-01']}, 'internal_lock-cache-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-02', 'producer_job': 'internal_lock-cache-02', 'marker': 'cache_shard_lock.json', 'suffixes': ['internal_lock-cache-02']}, 'internal_lock-cache-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-03', 'producer_job': 'internal_lock-cache-03', 'marker': 'cache_shard_lock.json', 'suffixes': ['internal_lock-cache-03']}, 'internal_lock-cache': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache', 'producer_job': 'internal_lock-cache', 'marker': 'cohort_manifest.json', 'suffixes': ['internal_lock-cache']}, '09b1': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-09b1', 'producer_job': '09b1', 'marker': 'cohort_manifest.json', 'suffixes': ['09b1/model-cache']}, '09c': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-09c', 'producer_job': '09c', 'marker': 'prediction_lock.json', 'suffixes': ['09c']}, 'internal_lock-truth': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-internal-lock-truth', 'producer_job': 'internal_lock-truth', 'marker': 'current_evaluator_manifest.json', 'suffixes': ['internal_lock-truth'], 'contains_outcomes': True}, '09d': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-09d', 'producer_job': '09d', 'marker': 'evaluation_lock.json', 'suffixes': ['09d'], 'contains_outcomes': True}, 'seen-rgb': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-rgb', 'producer_job': 'seen-rgb', 'marker': 'package_record.json', 'suffixes': ['seen-rgb/model-input']}, 'seen-regions-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-00', 'producer_job': 'seen-regions-00', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-00']}, 'seen-regions-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-01', 'producer_job': 'seen-regions-01', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-01']}, 'seen-regions-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-02', 'producer_job': 'seen-regions-02', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-02']}, 'seen-regions-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-03', 'producer_job': 'seen-regions-03', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-03']}, 'seen-regions-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-04', 'producer_job': 'seen-regions-04', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-04']}, 'seen-regions-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-05', 'producer_job': 'seen-regions-05', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-05']}, 'seen-regions-06': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-06', 'producer_job': 'seen-regions-06', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-06']}, 'seen-regions-07': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions-07', 'producer_job': 'seen-regions-07', 'marker': 'region_shard_lock.json', 'suffixes': ['seen-regions-07']}, 'seen-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-regions', 'producer_job': 'seen-regions', 'marker': 'artifact_lock.json', 'suffixes': ['seen-regions']}, 'seen-neutral': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-neutral', 'producer_job': 'seen-neutral', 'marker': 'neutral_correspondence.json', 'suffixes': ['seen-neutral']}, 'seen-cache-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-00', 'producer_job': 'seen-cache-00', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-00']}, 'seen-cache-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-01', 'producer_job': 'seen-cache-01', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-01']}, 'seen-cache-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-02', 'producer_job': 'seen-cache-02', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-02']}, 'seen-cache-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-03', 'producer_job': 'seen-cache-03', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-03']}, 'seen-cache-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-04', 'producer_job': 'seen-cache-04', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-04']}, 'seen-cache-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-05', 'producer_job': 'seen-cache-05', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-05']}, 'seen-cache-06': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-06', 'producer_job': 'seen-cache-06', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-06']}, 'seen-cache-07': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache-07', 'producer_job': 'seen-cache-07', 'marker': 'cache_shard_lock.json', 'suffixes': ['seen-cache-07']}, 'seen-cache': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-cache', 'producer_job': 'seen-cache', 'marker': 'cohort_manifest.json', 'suffixes': ['seen-cache']}, '10r-seen': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10r-seen', 'producer_job': '10r-seen', 'marker': 'cohort_manifest.json', 'suffixes': ['10r-seen/model-input']}, '10a-automatic': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10a-automatic', 'producer_job': '10a-automatic', 'marker': 'prediction_lock.json', 'suffixes': ['10a-automatic']}, '10a-assisted': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10a-assisted', 'producer_job': '10a-assisted', 'marker': 'prediction_lock.json', 'suffixes': ['10a-assisted']}, 'seen-truth': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-seen-truth', 'producer_job': 'seen-truth', 'marker': 'current_evaluator_manifest.json', 'suffixes': ['seen-truth'], 'contains_outcomes': True}, 'unseen-rgb': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-rgb', 'producer_job': 'unseen-rgb', 'marker': 'package_record.json', 'suffixes': ['unseen-rgb/model-input']}, 'unseen-regions-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-00', 'producer_job': 'unseen-regions-00', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-00']}, 'unseen-regions-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-01', 'producer_job': 'unseen-regions-01', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-01']}, 'unseen-regions-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-02', 'producer_job': 'unseen-regions-02', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-02']}, 'unseen-regions-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-03', 'producer_job': 'unseen-regions-03', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-03']}, 'unseen-regions-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-04', 'producer_job': 'unseen-regions-04', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-04']}, 'unseen-regions-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-05', 'producer_job': 'unseen-regions-05', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-05']}, 'unseen-regions-06': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-06', 'producer_job': 'unseen-regions-06', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-06']}, 'unseen-regions-07': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-07', 'producer_job': 'unseen-regions-07', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-07']}, 'unseen-regions-08': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-08', 'producer_job': 'unseen-regions-08', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-08']}, 'unseen-regions-09': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-09', 'producer_job': 'unseen-regions-09', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-09']}, 'unseen-regions-10': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-10', 'producer_job': 'unseen-regions-10', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-10']}, 'unseen-regions-11': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-11', 'producer_job': 'unseen-regions-11', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-11']}, 'unseen-regions-12': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-12', 'producer_job': 'unseen-regions-12', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-12']}, 'unseen-regions-13': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-13', 'producer_job': 'unseen-regions-13', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-13']}, 'unseen-regions-14': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-14', 'producer_job': 'unseen-regions-14', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-14']}, 'unseen-regions-15': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-15', 'producer_job': 'unseen-regions-15', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-15']}, 'unseen-regions-16': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-16', 'producer_job': 'unseen-regions-16', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-16']}, 'unseen-regions-17': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-17', 'producer_job': 'unseen-regions-17', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-17']}, 'unseen-regions-18': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-18', 'producer_job': 'unseen-regions-18', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-18']}, 'unseen-regions-19': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-19', 'producer_job': 'unseen-regions-19', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-19']}, 'unseen-regions-20': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-20', 'producer_job': 'unseen-regions-20', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-20']}, 'unseen-regions-21': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-21', 'producer_job': 'unseen-regions-21', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-21']}, 'unseen-regions-22': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-22', 'producer_job': 'unseen-regions-22', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-22']}, 'unseen-regions-23': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-23', 'producer_job': 'unseen-regions-23', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-23']}, 'unseen-regions-24': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions-24', 'producer_job': 'unseen-regions-24', 'marker': 'region_shard_lock.json', 'suffixes': ['unseen-regions-24']}, 'unseen-regions': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-regions', 'producer_job': 'unseen-regions', 'marker': 'artifact_lock.json', 'suffixes': ['unseen-regions']}, 'unseen-neutral': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-neutral', 'producer_job': 'unseen-neutral', 'marker': 'neutral_correspondence.json', 'suffixes': ['unseen-neutral']}, 'unseen-cache-00': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-00', 'producer_job': 'unseen-cache-00', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-00']}, 'unseen-cache-01': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-01', 'producer_job': 'unseen-cache-01', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-01']}, 'unseen-cache-02': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-02', 'producer_job': 'unseen-cache-02', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-02']}, 'unseen-cache-03': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-03', 'producer_job': 'unseen-cache-03', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-03']}, 'unseen-cache-04': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-04', 'producer_job': 'unseen-cache-04', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-04']}, 'unseen-cache-05': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-05', 'producer_job': 'unseen-cache-05', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-05']}, 'unseen-cache-06': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-06', 'producer_job': 'unseen-cache-06', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-06']}, 'unseen-cache-07': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-07', 'producer_job': 'unseen-cache-07', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-07']}, 'unseen-cache-08': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-08', 'producer_job': 'unseen-cache-08', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-08']}, 'unseen-cache-09': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-09', 'producer_job': 'unseen-cache-09', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-09']}, 'unseen-cache-10': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-10', 'producer_job': 'unseen-cache-10', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-10']}, 'unseen-cache-11': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-11', 'producer_job': 'unseen-cache-11', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-11']}, 'unseen-cache-12': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-12', 'producer_job': 'unseen-cache-12', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-12']}, 'unseen-cache-13': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-13', 'producer_job': 'unseen-cache-13', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-13']}, 'unseen-cache-14': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-14', 'producer_job': 'unseen-cache-14', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-14']}, 'unseen-cache-15': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-15', 'producer_job': 'unseen-cache-15', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-15']}, 'unseen-cache-16': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-16', 'producer_job': 'unseen-cache-16', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-16']}, 'unseen-cache-17': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-17', 'producer_job': 'unseen-cache-17', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-17']}, 'unseen-cache-18': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-18', 'producer_job': 'unseen-cache-18', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-18']}, 'unseen-cache-19': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-19', 'producer_job': 'unseen-cache-19', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-19']}, 'unseen-cache-20': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-20', 'producer_job': 'unseen-cache-20', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-20']}, 'unseen-cache-21': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-21', 'producer_job': 'unseen-cache-21', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-21']}, 'unseen-cache-22': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-22', 'producer_job': 'unseen-cache-22', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-22']}, 'unseen-cache-23': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-23', 'producer_job': 'unseen-cache-23', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-23']}, 'unseen-cache-24': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache-24', 'producer_job': 'unseen-cache-24', 'marker': 'cache_shard_lock.json', 'suffixes': ['unseen-cache-24']}, 'unseen-cache': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-cache', 'producer_job': 'unseen-cache', 'marker': 'cohort_manifest.json', 'suffixes': ['unseen-cache']}, '10r-unseen': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10r-unseen', 'producer_job': '10r-unseen', 'marker': 'cohort_manifest.json', 'suffixes': ['10r-unseen/model-input']}, '10a1-automatic': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10a1-automatic', 'producer_job': '10a1-automatic', 'marker': 'prediction_lock.json', 'suffixes': ['10a1-automatic']}, '10a1-assisted': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10a1-assisted', 'producer_job': '10a1-assisted', 'marker': 'prediction_lock.json', 'suffixes': ['10a1-assisted']}, 'unseen-truth': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-unseen-truth', 'producer_job': 'unseen-truth', 'marker': 'current_evaluator_manifest.json', 'suffixes': ['unseen-truth'], 'contains_outcomes': True}, '10a-finalize-automatic': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10a-finalize-automatic', 'producer_job': '10a-finalize-automatic', 'marker': 'summary_lock.json', 'suffixes': ['10a-finalize-automatic']}, '10a-finalize-assisted': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10a-finalize-assisted', 'producer_job': '10a-finalize-assisted', 'marker': 'summary_lock.json', 'suffixes': ['10a-finalize-assisted']}, '10b': {'kind': 'kernel', 'ref': 'esreekarreddy/vpi-current-10b', 'producer_job': '10b', 'marker': 'evaluation_lock.json', 'suffixes': ['10b'], 'contains_outcomes': True}}, 'jobs': [{'id': 'setup-cpu', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'runtime': 'cpu', 'setup_job': True, 'depends_on': [], 'config': {'profile': 'cpu', 'action': 'prepare', 'build_wheelhouse': True, 'wheelhouse': '/kaggle/working/setup-cpu/wheelhouse', 'target': '/tmp/vpi-runtime-setup-cpu', 'output_root': '/kaggle/working/setup-cpu', 'download_models': False}, 'attach_resources': []}, {'id': 'setup-dino', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'runtime': 'cpu', 'setup_job': True, 'depends_on': [], 'config': {'profile': 'dino', 'action': 'prepare', 'build_wheelhouse': True, 'wheelhouse': '/kaggle/working/setup-dino/wheelhouse', 'target': '/tmp/vpi-runtime-setup-dino', 'output_root': '/kaggle/working/setup-dino', 'download_models': True, 'model_asset_root': '/kaggle/working/setup-dino/model-assets', 'models_only': 'all'}, 'attach_resources': []}, {'id': 'setup-regions', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'runtime': 'cpu', 'setup_job': True, 'depends_on': [], 'config': {'profile': 'regions', 'action': 'prepare', 'build_wheelhouse': True, 'wheelhouse': '/kaggle/working/setup-regions/wheelhouse', 'target': '/tmp/vpi-runtime-setup-regions', 'output_root': '/kaggle/working/setup-regions', 'download_models': False}, 'attach_resources': []}, {'id': 'setup-tracking', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'runtime': 'cpu', 'setup_job': True, 'depends_on': [], 'config': {'profile': 'tracking', 'action': 'prepare', 'build_wheelhouse': True, 'wheelhouse': '/kaggle/working/setup-tracking/wheelhouse', 'target': '/tmp/vpi-runtime-setup-tracking', 'output_root': '/kaggle/working/setup-tracking', 'download_models': False}, 'attach_resources': []}, {'id': 'freeze', 'purpose': 'freeze', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu'], 'config': {'action': 'freeze', 'reference_root': {'$resource': 'support', 'path': 'method-reference'}, 'output_root': {'$output': 'freeze'}}}, {'id': 'development-index', 'purpose': 'rgb index', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu'], 'config': {'action': 'rgb_index', 'role': 'development', 'raw_root': {'$resource': 'raw'}, 'selection_root': {'$resource': 'selection'}, 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'development-index'}}}, {'id': 'development-geometry', 'purpose': 'raw geometry', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['development-index', 'setup-cpu'], 'config': {'action': 'raw_geometry', 'role': 'development', 'raw_root': {'$resource': 'raw'}, 'selection_root': {'$resource': 'selection'}, 'administrative_root': {'$resource': 'administrative'}, 'rgb_index_root': {'$resource': 'development-index'}, 'output_root': {'$output': 'development-geometry'}}}, {'id': 'calibration-index', 'purpose': 'rgb index', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu'], 'config': {'action': 'rgb_index', 'role': 'calibration', 'raw_root': {'$resource': 'raw'}, 'selection_root': {'$resource': 'selection'}, 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'calibration-index'}}}, {'id': 'calibration-geometry', 'purpose': 'raw geometry', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['calibration-index', 'setup-cpu'], 'config': {'action': 'raw_geometry', 'role': 'calibration', 'raw_root': {'$resource': 'raw'}, 'selection_root': {'$resource': 'selection'}, 'administrative_root': {'$resource': 'administrative'}, 'rgb_index_root': {'$resource': 'calibration-index'}, 'output_root': {'$output': 'calibration-geometry'}}}, {'id': 'internal_lock-index', 'purpose': 'rgb index', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu'], 'config': {'action': 'rgb_index', 'role': 'internal_lock', 'raw_root': {'$resource': 'raw'}, 'selection_root': {'$resource': 'selection'}, 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'internal_lock-index'}}}, {'id': 'internal_lock-geometry', 'purpose': 'raw geometry', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['internal_lock-index', 'setup-cpu'], 'config': {'action': 'raw_geometry', 'role': 'internal_lock', 'raw_root': {'$resource': 'raw'}, 'selection_root': {'$resource': 'selection'}, 'administrative_root': {'$resource': 'administrative'}, 'rgb_index_root': {'$resource': 'internal_lock-index'}, 'output_root': {'$output': 'internal_lock-geometry'}}}, {'id': 'seen-index', 'purpose': 'rgb index', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu'], 'config': {'action': 'rgb_index', 'role': 'seen', 'raw_root': {'$resource': 'raw'}, 'output_root': {'$output': 'seen-index'}}}, {'id': 'seen-geometry', 'purpose': 'raw geometry', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['seen-index', 'setup-cpu'], 'config': {'action': 'raw_geometry', 'role': 'seen', 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'seen-index'}, 'output_root': {'$output': 'seen-geometry'}}}, {'id': 'unseen-index', 'purpose': 'rgb index', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu'], 'config': {'action': 'rgb_index', 'role': 'unseen', 'raw_root': {'$resource': 'raw'}, 'output_root': {'$output': 'unseen-index'}}}, {'id': 'unseen-geometry', 'purpose': 'raw geometry', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['setup-cpu', 'unseen-index'], 'config': {'action': 'raw_geometry', 'role': 'unseen', 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'unseen-index'}, 'output_root': {'$output': 'unseen-geometry'}}}, {'id': 'official-exposure', 'purpose': 'official exposure', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['seen-geometry', 'setup-cpu', 'unseen-geometry'], 'config': {'action': 'official_exposure', 'index_roots': {'seen': {'$resource': 'seen-geometry'}, 'unseen': {'$resource': 'unseen-geometry'}}, 'policy': 'exposure_qualified_fixed_methods', 'output_root': {'$output': 'official-exposure'}}}, {'id': 'development-neutral', 'purpose': 'neutral', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['development-geometry', 'freeze', 'setup-cpu'], 'config': {'action': 'neutral', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'geometry_root': {'$resource': 'development-geometry'}, 'output_root': {'$output': 'development-neutral'}}}, {'id': 'development-cache-00', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-00-features'}, 'shard_count': 10, 'shard_index': 0, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-00'}}, 'compact_checkpoints': True}, {'id': 'development-cache-01', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-01-features'}, 'shard_count': 10, 'shard_index': 1, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-01'}}, 'compact_checkpoints': True}, {'id': 'development-cache-02', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-02-features'}, 'shard_count': 10, 'shard_index': 2, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-02'}}, 'compact_checkpoints': True}, {'id': 'development-cache-03', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-03-features'}, 'shard_count': 10, 'shard_index': 3, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-03'}}, 'compact_checkpoints': True}, {'id': 'development-cache-04', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-04-features'}, 'shard_count': 10, 'shard_index': 4, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-04'}}, 'compact_checkpoints': True}, {'id': 'development-cache-05', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-05-features'}, 'shard_count': 10, 'shard_index': 5, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-05'}}, 'compact_checkpoints': True}, {'id': 'development-cache-06', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-06-features'}, 'shard_count': 10, 'shard_index': 6, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-06'}}, 'compact_checkpoints': True}, {'id': 'development-cache-07', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-07-features'}, 'shard_count': 10, 'shard_index': 7, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-07'}}, 'compact_checkpoints': True}, {'id': 'development-cache-08', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-08-features'}, 'shard_count': 10, 'shard_index': 8, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-08'}}, 'compact_checkpoints': True}, {'id': 'development-cache-09', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'native-development-components', 'components': {'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'development-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'development-cache-09-features'}, 'shard_count': 10, 'shard_index': 9, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'feature_roots': [{'$resource': 'reuse100'}], 'output_root': {'$output': 'development-cache-09'}}, 'compact_checkpoints': True}, {'id': 'development-cache', 'purpose': 'cache merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['development-cache-00', 'development-cache-01', 'development-cache-02', 'development-cache-03', 'development-cache-04', 'development-cache-05', 'development-cache-06', 'development-cache-07', 'development-cache-08', 'development-cache-09', 'freeze', 'setup-cpu'], 'config': {'action': 'cache_merge', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_shards': [{'$resource': 'development-cache-00'}, {'$resource': 'development-cache-01'}, {'$resource': 'development-cache-02'}, {'$resource': 'development-cache-03'}, {'$resource': 'development-cache-04'}, {'$resource': 'development-cache-05'}, {'$resource': 'development-cache-06'}, {'$resource': 'development-cache-07'}, {'$resource': 'development-cache-08'}, {'$resource': 'development-cache-09'}], 'maximum_output_bytes': 19327352832, 'output_root': {'$output': 'development-cache'}}}, {'id': 'development-predict', 'purpose': 'dev-predict', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['development-cache', 'freeze', 'setup-cpu'], 'config': {'action': 'dev-predict', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_root': {'$resource': 'development-cache'}, 'output_root': {'$output': 'development-predict'}}}, {'id': 'development-truth', 'purpose': 'raw truth', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['development-cache', 'development-geometry', 'development-index', 'development-predict', 'freeze', 'setup-cpu'], 'config': {'action': 'raw_truth', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'raw_root': {'$resource': 'raw'}, 'administrative_root': {'$resource': 'administrative'}, 'rgb_index_root': {'$resource': 'development-index'}, 'geometry_root': {'$resource': 'development-geometry'}, 'cache_root': {'$resource': 'development-cache'}, 'prediction_root': {'$resource': 'development-predict'}, 'output_root': {'$output': 'development-truth'}}}, {'id': 'development-evaluate', 'purpose': 'dev-evaluate', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['development-cache', 'development-predict', 'development-truth', 'freeze', 'setup-cpu', 'setup-tracking'], 'config': {'action': 'dev-evaluate', 'role': 'development', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_root': {'$resource': 'development-cache'}, 'prediction_root': {'$resource': 'development-predict'}, 'evaluator_root': {'$resource': 'development-truth'}, 'report_root': {'$output': 'development-evaluate-report'}, 'output_root': {'$output': 'development-evaluate'}, 'publish_tracking': True}, 'attach_resources': ['runtime-tracking', 'wheels-tracking']}, {'id': 'calibration-rgb', 'purpose': 'rgb release', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['calibration-geometry', 'calibration-index', 'development-evaluate', 'freeze', 'setup-cpu'], 'config': {'action': 'rgb_release', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'calibration-index'}, 'maximum_new_scenes': 600, 'output_root': {'$output': 'calibration-rgb'}}}, {'id': 'calibration-regions-00', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 0, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'calibration-regions-00'}}}, {'id': 'calibration-regions-01', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 1, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'calibration-regions-01'}}}, {'id': 'calibration-regions-02', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 2, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'calibration-regions-02'}}}, {'id': 'calibration-regions-03', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 3, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'calibration-regions-03'}}}, {'id': 'calibration-regions-04', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 4, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'calibration-regions-04'}}}, {'id': 'calibration-regions-05', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 5, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'calibration-regions-05'}}}, {'id': 'calibration-regions', 'purpose': 'region merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-regions-00', 'calibration-regions-01', 'calibration-regions-02', 'calibration-regions-03', 'calibration-regions-04', 'calibration-regions-05', 'calibration-rgb', 'freeze', 'setup-cpu'], 'config': {'action': 'region_merge', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'region_shards': [{'$resource': 'calibration-regions-00'}, {'$resource': 'calibration-regions-01'}, {'$resource': 'calibration-regions-02'}, {'$resource': 'calibration-regions-03'}, {'$resource': 'calibration-regions-04'}, {'$resource': 'calibration-regions-05'}], 'output_root': {'$output': 'calibration-regions'}}}, {'id': 'calibration-neutral', 'purpose': 'neutral', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['calibration-geometry', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-cpu'], 'config': {'action': 'neutral', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'geometry_root': {'$resource': 'calibration-geometry'}, 'output_root': {'$output': 'calibration-neutral'}}}, {'id': 'calibration-cache-00', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'calibration-cache-00-features'}, 'shard_count': 6, 'shard_index': 0, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'calibration-cache-00'}}, 'compact_checkpoints': True}, {'id': 'calibration-cache-01', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'calibration-cache-01-features'}, 'shard_count': 6, 'shard_index': 1, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'calibration-cache-01'}}, 'compact_checkpoints': True}, {'id': 'calibration-cache-02', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'calibration-cache-02-features'}, 'shard_count': 6, 'shard_index': 2, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'calibration-cache-02'}}, 'compact_checkpoints': True}, {'id': 'calibration-cache-03', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'calibration-cache-03-features'}, 'shard_count': 6, 'shard_index': 3, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'calibration-cache-03'}}, 'compact_checkpoints': True}, {'id': 'calibration-cache-04', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'calibration-cache-04-features'}, 'shard_count': 6, 'shard_index': 4, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'calibration-cache-04'}}, 'compact_checkpoints': True}, {'id': 'calibration-cache-05', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'calibration-cache-05-features'}, 'shard_count': 6, 'shard_index': 5, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'calibration-cache-05'}}, 'compact_checkpoints': True}, {'id': 'calibration-cache', 'purpose': 'cache merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-cache-00', 'calibration-cache-01', 'calibration-cache-02', 'calibration-cache-03', 'calibration-cache-04', 'calibration-cache-05', 'freeze', 'setup-cpu'], 'config': {'action': 'cache_merge', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_shards': [{'$resource': 'calibration-cache-00'}, {'$resource': 'calibration-cache-01'}, {'$resource': 'calibration-cache-02'}, {'$resource': 'calibration-cache-03'}, {'$resource': 'calibration-cache-04'}, {'$resource': 'calibration-cache-05'}], 'maximum_output_bytes': 19327352832, 'output_root': {'$output': 'calibration-cache'}}}, {'id': '08f1', 'purpose': 'release', 'notebook': 'notebooks/protected-evaluation/08f1-toysad8k-calibration-resource-release.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['calibration-cache', 'freeze', 'setup-cpu'], 'config': {'action': 'release', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_root': {'$resource': 'calibration-cache'}, 'output_root': {'$output': '08f1'}}}, {'id': '09a', 'purpose': 'predict', 'notebook': 'notebooks/protected-evaluation/09a-toysad8k-protected-calibration-inference.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['08f1', 'freeze', 'setup-cpu'], 'config': {'action': 'predict', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_root': {'$resource': '08f1'}, 'output_root': {'$output': '09a'}}}, {'id': 'calibration-truth', 'purpose': 'raw truth', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['08f1', '09a', 'calibration-geometry', 'calibration-index', 'freeze', 'setup-cpu'], 'config': {'action': 'raw_truth', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'raw_root': {'$resource': 'raw'}, 'administrative_root': {'$resource': 'administrative'}, 'rgb_index_root': {'$resource': 'calibration-index'}, 'geometry_root': {'$resource': 'calibration-geometry'}, 'cache_root': {'$resource': '08f1'}, 'prediction_root': {'$resource': '09a'}, 'output_root': {'$output': 'calibration-truth'}}}, {'id': '09b', 'purpose': 'evaluate', 'notebook': 'notebooks/protected-evaluation/09b-toysad8k-calibration-review-policy.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['08f1', '09a', 'calibration-truth', 'freeze', 'setup-cpu', 'setup-tracking'], 'config': {'action': 'evaluate', 'role': 'calibration', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'cache_root': {'$resource': '08f1'}, 'prediction_root': {'$resource': '09a'}, 'evaluator_root': {'$resource': 'calibration-truth'}, 'report_root': {'$output': '09b-report'}, 'output_root': {'$output': '09b'}, 'publish_tracking': True}, 'attach_resources': ['runtime-tracking', 'wheels-tracking']}, {'id': 'internal_lock-rgb', 'purpose': 'rgb release', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09b', 'freeze', 'internal_lock-geometry', 'internal_lock-index', 'setup-cpu'], 'config': {'action': 'rgb_release', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'internal_lock-index'}, 'maximum_new_scenes': 400, 'output_root': {'$output': 'internal_lock-rgb'}}}, {'id': 'internal_lock-regions-00', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 0, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'internal_lock-regions-00'}}}, {'id': 'internal_lock-regions-01', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 1, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'internal_lock-regions-01'}}}, {'id': 'internal_lock-regions-02', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 2, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'internal_lock-regions-02'}}}, {'id': 'internal_lock-regions-03', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 3, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'internal_lock-regions-03'}}}, {'id': 'internal_lock-regions', 'purpose': 'region merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-regions-00', 'internal_lock-regions-01', 'internal_lock-regions-02', 'internal_lock-regions-03', 'internal_lock-rgb', 'setup-cpu'], 'config': {'action': 'region_merge', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'region_shards': [{'$resource': 'internal_lock-regions-00'}, {'$resource': 'internal_lock-regions-01'}, {'$resource': 'internal_lock-regions-02'}, {'$resource': 'internal_lock-regions-03'}], 'output_root': {'$output': 'internal_lock-regions'}}}, {'id': 'internal_lock-neutral', 'purpose': 'neutral', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09b', 'freeze', 'internal_lock-geometry', 'internal_lock-regions', 'internal_lock-rgb', 'setup-cpu'], 'config': {'action': 'neutral', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'geometry_root': {'$resource': 'internal_lock-geometry'}, 'output_root': {'$output': 'internal_lock-neutral'}}}, {'id': 'internal_lock-cache-00', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'internal_lock-cache-00-features'}, 'shard_count': 4, 'shard_index': 0, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'internal_lock-cache-00'}}, 'compact_checkpoints': True}, {'id': 'internal_lock-cache-01', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'internal_lock-cache-01-features'}, 'shard_count': 4, 'shard_index': 1, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'internal_lock-cache-01'}}, 'compact_checkpoints': True}, {'id': 'internal_lock-cache-02', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'internal_lock-cache-02-features'}, 'shard_count': 4, 'shard_index': 2, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'internal_lock-cache-02'}}, 'compact_checkpoints': True}, {'id': 'internal_lock-cache-03', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'internal_lock-cache-03-features'}, 'shard_count': 4, 'shard_index': 3, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'internal_lock-cache-03'}}, 'compact_checkpoints': True}, {'id': 'internal_lock-cache', 'purpose': 'cache merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-cache-00', 'internal_lock-cache-01', 'internal_lock-cache-02', 'internal_lock-cache-03', 'setup-cpu'], 'config': {'action': 'cache_merge', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'cache_shards': [{'$resource': 'internal_lock-cache-00'}, {'$resource': 'internal_lock-cache-01'}, {'$resource': 'internal_lock-cache-02'}, {'$resource': 'internal_lock-cache-03'}], 'maximum_output_bytes': 19327352832, 'output_root': {'$output': 'internal_lock-cache'}}}, {'id': '09b1', 'purpose': 'release', 'notebook': 'notebooks/protected-evaluation/09b1-toysad8k-internal-lock-resource-release.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', 'freeze', 'internal_lock-cache', 'setup-cpu'], 'config': {'action': 'release', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'cache_root': {'$resource': 'internal_lock-cache'}, 'output_root': {'$output': '09b1'}}}, {'id': '09c', 'purpose': 'predict', 'notebook': 'notebooks/protected-evaluation/09c-toysad8k-protected-internal-lock-inference.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09b', '09b1', 'freeze', 'setup-cpu'], 'config': {'action': 'predict', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'cache_root': {'$resource': '09b1'}, 'output_root': {'$output': '09c'}}}, {'id': 'internal_lock-truth', 'purpose': 'raw truth', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09b', '09b1', '09c', 'freeze', 'internal_lock-geometry', 'internal_lock-index', 'setup-cpu'], 'config': {'action': 'raw_truth', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'raw_root': {'$resource': 'raw'}, 'administrative_root': {'$resource': 'administrative'}, 'rgb_index_root': {'$resource': 'internal_lock-index'}, 'geometry_root': {'$resource': 'internal_lock-geometry'}, 'cache_root': {'$resource': '09b1'}, 'prediction_root': {'$resource': '09c'}, 'output_root': {'$output': 'internal_lock-truth'}}}, {'id': '09d', 'purpose': 'evaluate', 'notebook': 'notebooks/protected-evaluation/09d-toysad8k-internal-lock-one-shot-evaluation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09b', '09b1', '09c', 'freeze', 'internal_lock-truth', 'setup-cpu', 'setup-tracking'], 'config': {'action': 'evaluate', 'role': 'internal_lock', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'calibration_receipt_root': {'$resource': '09b'}, 'cache_root': {'$resource': '09b1'}, 'prediction_root': {'$resource': '09c'}, 'evaluator_root': {'$resource': 'internal_lock-truth'}, 'report_root': {'$output': '09d-report'}, 'output_root': {'$output': '09d'}, 'publish_tracking': True}, 'attach_resources': ['runtime-tracking', 'wheels-tracking']}, {'id': 'seen-rgb', 'purpose': 'rgb release', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-geometry', 'seen-index', 'setup-cpu'], 'config': {'action': 'rgb_release', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'seen-index'}, 'maximum_new_scenes': 715, 'output_root': {'$output': 'seen-rgb'}}}, {'id': 'seen-regions-00', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 0, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-00'}}}, {'id': 'seen-regions-01', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 1, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-01'}}}, {'id': 'seen-regions-02', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 2, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-02'}}}, {'id': 'seen-regions-03', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 3, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-03'}}}, {'id': 'seen-regions-04', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 4, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-04'}}}, {'id': 'seen-regions-05', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 5, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-05'}}}, {'id': 'seen-regions-06', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 6, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-06'}}}, {'id': 'seen-regions-07', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'config': {'action': 'extract', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 7, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'seen-regions-07'}}}, {'id': 'seen-regions', 'purpose': 'region merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-regions-00', 'seen-regions-01', 'seen-regions-02', 'seen-regions-03', 'seen-regions-04', 'seen-regions-05', 'seen-regions-06', 'seen-regions-07', 'seen-rgb', 'setup-cpu'], 'config': {'action': 'region_merge', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'region_shards': [{'$resource': 'seen-regions-00'}, {'$resource': 'seen-regions-01'}, {'$resource': 'seen-regions-02'}, {'$resource': 'seen-regions-03'}, {'$resource': 'seen-regions-04'}, {'$resource': 'seen-regions-05'}, {'$resource': 'seen-regions-06'}, {'$resource': 'seen-regions-07'}], 'output_root': {'$output': 'seen-regions'}}}, {'id': 'seen-neutral', 'purpose': 'neutral', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-geometry', 'seen-regions', 'seen-rgb', 'setup-cpu'], 'config': {'action': 'neutral', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'geometry_root': {'$resource': 'seen-geometry'}, 'output_root': {'$output': 'seen-neutral'}}}, {'id': 'seen-cache-00', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-00-features'}, 'shard_count': 8, 'shard_index': 0, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-00'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-01', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-01-features'}, 'shard_count': 8, 'shard_index': 1, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-01'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-02', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-02-features'}, 'shard_count': 8, 'shard_index': 2, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-02'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-03', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-03-features'}, 'shard_count': 8, 'shard_index': 3, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-03'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-04', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-04-features'}, 'shard_count': 8, 'shard_index': 4, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-04'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-05', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-05-features'}, 'shard_count': 8, 'shard_index': 5, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-05'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-06', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-06-features'}, 'shard_count': 8, 'shard_index': 6, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-06'}}, 'compact_checkpoints': True}, {'id': 'seen-cache-07', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'config': {'action': 'prepare_shard', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'seen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'seen-cache-07-features'}, 'shard_count': 8, 'shard_index': 7, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'seen-cache-07'}}, 'compact_checkpoints': True}, {'id': 'seen-cache', 'purpose': 'cache merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-cache-00', 'seen-cache-01', 'seen-cache-02', 'seen-cache-03', 'seen-cache-04', 'seen-cache-05', 'seen-cache-06', 'seen-cache-07', 'setup-cpu'], 'config': {'action': 'cache_merge', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'cache_shards': [{'$resource': 'seen-cache-00'}, {'$resource': 'seen-cache-01'}, {'$resource': 'seen-cache-02'}, {'$resource': 'seen-cache-03'}, {'$resource': 'seen-cache-04'}, {'$resource': 'seen-cache-05'}, {'$resource': 'seen-cache-06'}, {'$resource': 'seen-cache-07'}], 'maximum_output_bytes': 19327352832, 'output_root': {'$output': 'seen-cache'}}}, {'id': '10r-seen', 'purpose': 'release', 'notebook': 'notebooks/official-evaluation/10r-toysad8k-official-release.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-cache', 'setup-cpu'], 'config': {'action': 'release', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'cache_root': {'$resource': 'seen-cache'}, 'output_root': {'$output': '10r-seen'}}}, {'id': '10a-automatic', 'purpose': 'predict', 'notebook': 'notebooks/official-evaluation/10a-toysad8k-official-seen-frozen-inference.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', '10r-seen', 'freeze', 'official-exposure', 'setup-cpu'], 'config': {'action': 'predict', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'mode': 'automatic', 'cache_root': {'$resource': '10r-seen'}, 'output_root': {'$output': '10a-automatic'}}}, {'id': '10a-assisted', 'purpose': 'predict', 'notebook': 'notebooks/official-evaluation/10a-toysad8k-official-seen-frozen-inference.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', '10r-seen', 'freeze', 'official-exposure', 'setup-cpu'], 'config': {'action': 'predict', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'mode': 'assisted', 'cache_root': {'$resource': '10r-seen'}, 'output_root': {'$output': '10a-assisted'}}}, {'id': 'seen-truth', 'purpose': 'raw truth', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', '10a-assisted', '10a-automatic', '10r-seen', 'freeze', 'official-exposure', 'seen-geometry', 'seen-index', 'setup-cpu'], 'config': {'action': 'raw_truth', 'role': 'seen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'seen-index'}, 'geometry_root': {'$resource': 'seen-geometry'}, 'cache_root': {'$resource': '10r-seen'}, 'prediction_roots': {'automatic': {'$resource': '10a-automatic'}, 'assisted': {'$resource': '10a-assisted'}}, 'output_root': {'$output': 'seen-truth'}}}, {'id': 'unseen-rgb', 'purpose': 'rgb release', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-geometry', 'unseen-index'], 'config': {'action': 'rgb_release', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'unseen-index'}, 'maximum_new_scenes': 2437, 'output_root': {'$output': 'unseen-rgb'}}}, {'id': 'unseen-regions-00', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 0, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-00'}}}, {'id': 'unseen-regions-01', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 1, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-01'}}}, {'id': 'unseen-regions-02', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 2, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-02'}}}, {'id': 'unseen-regions-03', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 3, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-03'}}}, {'id': 'unseen-regions-04', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 4, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-04'}}}, {'id': 'unseen-regions-05', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 5, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-05'}}}, {'id': 'unseen-regions-06', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 6, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-06'}}}, {'id': 'unseen-regions-07', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 7, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-07'}}}, {'id': 'unseen-regions-08', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 8, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-08'}}}, {'id': 'unseen-regions-09', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 9, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-09'}}}, {'id': 'unseen-regions-10', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 10, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-10'}}}, {'id': 'unseen-regions-11', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 11, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-11'}}}, {'id': 'unseen-regions-12', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 12, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-12'}}}, {'id': 'unseen-regions-13', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 13, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-13'}}}, {'id': 'unseen-regions-14', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 14, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-14'}}}, {'id': 'unseen-regions-15', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 15, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-15'}}}, {'id': 'unseen-regions-16', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 16, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-16'}}}, {'id': 'unseen-regions-17', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 17, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-17'}}}, {'id': 'unseen-regions-18', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 18, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-18'}}}, {'id': 'unseen-regions-19', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 19, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-19'}}}, {'id': 'unseen-regions-20', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 20, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-20'}}}, {'id': 'unseen-regions-21', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 21, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-21'}}}, {'id': 'unseen-regions-22', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 22, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-22'}}}, {'id': 'unseen-regions-23', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 23, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-23'}}}, {'id': 'unseen-regions-24', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'runtime': 'regions-gpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'config': {'action': 'extract', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 24, 'maximum_new_views': 2000, 'attempt_timeout_seconds': 600, 'output_root': {'$output': 'unseen-regions-24'}}}, {'id': 'unseen-regions', 'purpose': 'region merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-regions-00', 'unseen-regions-01', 'unseen-regions-02', 'unseen-regions-03', 'unseen-regions-04', 'unseen-regions-05', 'unseen-regions-06', 'unseen-regions-07', 'unseen-regions-08', 'unseen-regions-09', 'unseen-regions-10', 'unseen-regions-11', 'unseen-regions-12', 'unseen-regions-13', 'unseen-regions-14', 'unseen-regions-15', 'unseen-regions-16', 'unseen-regions-17', 'unseen-regions-18', 'unseen-regions-19', 'unseen-regions-20', 'unseen-regions-21', 'unseen-regions-22', 'unseen-regions-23', 'unseen-regions-24', 'unseen-rgb'], 'config': {'action': 'region_merge', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'region_shards': [{'$resource': 'unseen-regions-00'}, {'$resource': 'unseen-regions-01'}, {'$resource': 'unseen-regions-02'}, {'$resource': 'unseen-regions-03'}, {'$resource': 'unseen-regions-04'}, {'$resource': 'unseen-regions-05'}, {'$resource': 'unseen-regions-06'}, {'$resource': 'unseen-regions-07'}, {'$resource': 'unseen-regions-08'}, {'$resource': 'unseen-regions-09'}, {'$resource': 'unseen-regions-10'}, {'$resource': 'unseen-regions-11'}, {'$resource': 'unseen-regions-12'}, {'$resource': 'unseen-regions-13'}, {'$resource': 'unseen-regions-14'}, {'$resource': 'unseen-regions-15'}, {'$resource': 'unseen-regions-16'}, {'$resource': 'unseen-regions-17'}, {'$resource': 'unseen-regions-18'}, {'$resource': 'unseen-regions-19'}, {'$resource': 'unseen-regions-20'}, {'$resource': 'unseen-regions-21'}, {'$resource': 'unseen-regions-22'}, {'$resource': 'unseen-regions-23'}, {'$resource': 'unseen-regions-24'}], 'output_root': {'$output': 'unseen-regions'}}}, {'id': 'unseen-neutral', 'purpose': 'neutral', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-geometry', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'neutral', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'geometry_root': {'$resource': 'unseen-geometry'}, 'output_root': {'$output': 'unseen-neutral'}}}, {'id': 'unseen-cache-00', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-00-features'}, 'shard_count': 25, 'shard_index': 0, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-00'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-01', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-01-features'}, 'shard_count': 25, 'shard_index': 1, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-01'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-02', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-02-features'}, 'shard_count': 25, 'shard_index': 2, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-02'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-03', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-03-features'}, 'shard_count': 25, 'shard_index': 3, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-03'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-04', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-04-features'}, 'shard_count': 25, 'shard_index': 4, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-04'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-05', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-05-features'}, 'shard_count': 25, 'shard_index': 5, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-05'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-06', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-06-features'}, 'shard_count': 25, 'shard_index': 6, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-06'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-07', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-07-features'}, 'shard_count': 25, 'shard_index': 7, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-07'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-08', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-08-features'}, 'shard_count': 25, 'shard_index': 8, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-08'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-09', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-09-features'}, 'shard_count': 25, 'shard_index': 9, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-09'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-10', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-10-features'}, 'shard_count': 25, 'shard_index': 10, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-10'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-11', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-11-features'}, 'shard_count': 25, 'shard_index': 11, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-11'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-12', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-12-features'}, 'shard_count': 25, 'shard_index': 12, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-12'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-13', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-13-features'}, 'shard_count': 25, 'shard_index': 13, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-13'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-14', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-14-features'}, 'shard_count': 25, 'shard_index': 14, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-14'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-15', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-15-features'}, 'shard_count': 25, 'shard_index': 15, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-15'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-16', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-16-features'}, 'shard_count': 25, 'shard_index': 16, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-16'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-17', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-17-features'}, 'shard_count': 25, 'shard_index': 17, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-17'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-18', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-18-features'}, 'shard_count': 25, 'shard_index': 18, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-18'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-19', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-19-features'}, 'shard_count': 25, 'shard_index': 19, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-19'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-20', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-20-features'}, 'shard_count': 25, 'shard_index': 20, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-20'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-21', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-21-features'}, 'shard_count': 25, 'shard_index': 21, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-21'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-22', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-22-features'}, 'shard_count': 25, 'shard_index': 22, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-22'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-23', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-23-features'}, 'shard_count': 25, 'shard_index': 23, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-23'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache-24', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'runtime': 'dino-cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'config': {'action': 'prepare_shard', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'feature_root': {'$output': 'unseen-cache-24-features'}, 'shard_count': 25, 'shard_index': 24, 'max_new_units': 2000, 'max_new_scenes': 100, 'maximum_output_bytes': 16106127360, 'output_root': {'$output': 'unseen-cache-24'}}, 'compact_checkpoints': True}, {'id': 'unseen-cache', 'purpose': 'cache merge', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-cache-00', 'unseen-cache-01', 'unseen-cache-02', 'unseen-cache-03', 'unseen-cache-04', 'unseen-cache-05', 'unseen-cache-06', 'unseen-cache-07', 'unseen-cache-08', 'unseen-cache-09', 'unseen-cache-10', 'unseen-cache-11', 'unseen-cache-12', 'unseen-cache-13', 'unseen-cache-14', 'unseen-cache-15', 'unseen-cache-16', 'unseen-cache-17', 'unseen-cache-18', 'unseen-cache-19', 'unseen-cache-20', 'unseen-cache-21', 'unseen-cache-22', 'unseen-cache-23', 'unseen-cache-24'], 'config': {'action': 'cache_merge', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'cache_shards': [{'$resource': 'unseen-cache-00'}, {'$resource': 'unseen-cache-01'}, {'$resource': 'unseen-cache-02'}, {'$resource': 'unseen-cache-03'}, {'$resource': 'unseen-cache-04'}, {'$resource': 'unseen-cache-05'}, {'$resource': 'unseen-cache-06'}, {'$resource': 'unseen-cache-07'}, {'$resource': 'unseen-cache-08'}, {'$resource': 'unseen-cache-09'}, {'$resource': 'unseen-cache-10'}, {'$resource': 'unseen-cache-11'}, {'$resource': 'unseen-cache-12'}, {'$resource': 'unseen-cache-13'}, {'$resource': 'unseen-cache-14'}, {'$resource': 'unseen-cache-15'}, {'$resource': 'unseen-cache-16'}, {'$resource': 'unseen-cache-17'}, {'$resource': 'unseen-cache-18'}, {'$resource': 'unseen-cache-19'}, {'$resource': 'unseen-cache-20'}, {'$resource': 'unseen-cache-21'}, {'$resource': 'unseen-cache-22'}, {'$resource': 'unseen-cache-23'}, {'$resource': 'unseen-cache-24'}], 'maximum_output_bytes': 19327352832, 'output_root': {'$output': 'unseen-cache'}}}, {'id': '10r-unseen', 'purpose': 'release', 'notebook': 'notebooks/official-evaluation/10r-toysad8k-official-release.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-cache'], 'config': {'action': 'release', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'cache_root': {'$resource': 'unseen-cache'}, 'output_root': {'$output': '10r-unseen'}}}, {'id': '10a1-automatic', 'purpose': 'predict', 'notebook': 'notebooks/official-evaluation/10a1-toysad8k-official-unseen-frozen-inference.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'config': {'action': 'predict', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'mode': 'automatic', 'cache_root': {'$resource': '10r-unseen'}, 'output_root': {'$output': '10a1-automatic'}}}, {'id': '10a1-assisted', 'purpose': 'predict', 'notebook': 'notebooks/official-evaluation/10a1-toysad8k-official-unseen-frozen-inference.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'config': {'action': 'predict', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'mode': 'assisted', 'cache_root': {'$resource': '10r-unseen'}, 'output_root': {'$output': '10a1-assisted'}}}, {'id': 'unseen-truth', 'purpose': 'raw truth', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', '10a1-assisted', '10a1-automatic', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-geometry', 'unseen-index'], 'config': {'action': 'raw_truth', 'role': 'unseen', 'method_freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'unseen-index'}, 'geometry_root': {'$resource': 'unseen-geometry'}, 'cache_root': {'$resource': '10r-unseen'}, 'prediction_roots': {'automatic': {'$resource': '10a1-automatic'}, 'assisted': {'$resource': '10a1-assisted'}}, 'output_root': {'$output': 'unseen-truth'}}}, {'id': '10a-finalize-automatic', 'purpose': 'finalize', 'notebook': 'notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', '10a-automatic', '10a1-automatic', '10r-seen', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'config': {'action': 'finalize', 'freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'mode': 'automatic', 'prediction_roots': {'seen': {'$resource': '10a-automatic'}, 'unseen': {'$resource': '10a1-automatic'}}, 'cache_roots': {'seen': {'$resource': '10r-seen'}, 'unseen': {'$resource': '10r-unseen'}}, 'output_root': {'$output': '10a-finalize-automatic'}}}, {'id': '10a-finalize-assisted', 'purpose': 'finalize', 'notebook': 'notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb', 'runtime': 'cpu', 'input_boundary': 'model_only', 'depends_on': ['09d', '10a-assisted', '10a1-assisted', '10r-seen', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'config': {'action': 'finalize', 'freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'mode': 'assisted', 'prediction_roots': {'seen': {'$resource': '10a-assisted'}, 'unseen': {'$resource': '10a1-assisted'}}, 'cache_roots': {'seen': {'$resource': '10r-seen'}, 'unseen': {'$resource': '10r-unseen'}}, 'output_root': {'$output': '10a-finalize-assisted'}}}, {'id': '10b', 'purpose': 'evaluate', 'notebook': 'notebooks/official-evaluation/10b-toysad8k-official-seen-unseen-evaluation.ipynb', 'runtime': 'cpu', 'input_boundary': 'trusted_preparation', 'depends_on': ['09d', '10a-assisted', '10a-automatic', '10a-finalize-assisted', '10a-finalize-automatic', '10a1-assisted', '10a1-automatic', '10r-seen', '10r-unseen', 'freeze', 'official-exposure', 'seen-truth', 'setup-cpu', 'setup-tracking', 'unseen-truth'], 'config': {'action': 'evaluate', 'freeze_root': {'$resource': 'freeze'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'decision_root': {'$resource': '09d'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'prediction_roots': {'automatic': {'seen': {'$resource': '10a-automatic'}, 'unseen': {'$resource': '10a1-automatic'}}, 'assisted': {'seen': {'$resource': '10a-assisted'}, 'unseen': {'$resource': '10a1-assisted'}}}, 'prediction_summary_roots': {'automatic': {'$resource': '10a-finalize-automatic'}, 'assisted': {'$resource': '10a-finalize-assisted'}}, 'cache_roots': {'seen': {'$resource': '10r-seen'}, 'unseen': {'$resource': '10r-unseen'}}, 'evaluator_roots': {'seen': {'$resource': 'seen-truth'}, 'unseen': {'$resource': 'unseen-truth'}}, 'report_root': {'$output': '10b-report'}, 'output_root': {'$output': '10b'}, 'publish_tracking': True}, 'attach_resources': ['runtime-tracking', 'wheels-tracking']}], 'scientific_runs_started': False, 'queue_note': 'Repeated cache jobs are100-scene resource shards of fixed methods, not new scoring experiments.'}, 'setup-regions', SOURCE['snapshot_sha256'], tracking=globals().get('LAUNCH_TRACKING'))
print(JOB_COMPLETION)

{'schema': 'vpi-current-job-completion/v1', 'status': 'outputs_complete', 'job_id': 'setup-regions', 'plan_sha256': 'd5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49', 'source_snapshot_sha256': 'ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf', 'outputs': {'runtime-regions': '7ec1dcbf08da8f8be23107ff84c81e85cd47b4977fcb8259353790573c107b35', 'wheels-regions': '2e3ed8c752ee402304cba53476fda5f9b3e49d5968ad51f148fb9844d83b919a'}, 'full_download_verified': False}
